# Installed Base Reliability and Service Cost Forecasting
## Part 2: Survival Modeling

### Where we are
Part 1 produced `component_lifetimes.csv`: one row per component lifetime, with how many days it lasted and whether it ended in a **failure** (`event = 1`) or was **censored** (`event = 0`, still working when it was swapped preventively or when the data ended).

Part 1 also computed a crude AFR (Annualized Failure Rate) per component. That number assumes the failure rate is **constant over a part's life**. Real parts rarely behave that way, and the difference matters for service cost.

### The questions this notebook answers
1. **How long do components survive?** Kaplan Meier curves, which handle censoring without assuming any distribution.
2. **What shape does failure take?** 2 and 3 parameter Weibull fits that tell us whether parts die young, fail randomly, or wear out, and whether there is a failure free period.
3. **What drives failure risk?** Cox PH (Proportional Hazards) regression on machine age, model (platform), and usage.
4. **What does that mean for next quarter?** Conditional failure probabilities for the units installed today, the bridge into Part 3.

### Why the shape of failure changes the cost forecast (numerical example)
Take a component with Weibull shape β = 1.4 and scale η = 300 days (β above 1 means wear out). The chance it fails in the next 90 days depends on how old it already is:

| Current age | P(survived to this age) | P(fails in next 90 days) |
|---|---|---|
| 0 days (new) | 1.000 | 16.9% |
| 100 days | 0.807 | 26.9% |
| 250 days | 0.461 | 34.1% |

A constant rate model gives the **same** probability for all three. On a fleet skewed toward older parts it underforecasts failures; on a freshly refurbished fleet it overforecasts. The Weibull shape is what lets the forecast see the age mix of the installed base.

### Key vocabulary
| Term | Meaning | Service interpretation |
|---|---|---|
| Survival S(t) | probability a part is still working at age t | share of installed parts still alive |
| Hazard h(t) | instantaneous failure rate at age t, given survival to t | how risky a part of this age is right now |
| Censoring | we know the part lasted *at least* t | preventive swaps, parts still running |
| Left truncation | we only see a part because it survived to the start of the data | parts installed before records began |
| Hazard ratio | multiplier on the hazard from a covariate | "model4 tools fail 1.6x as often at every age" |

### Project roadmap
| Part | Notebook | Output |
|---|---|---|
| 1 | Data foundation and failure behavior | Clean component lifetime table with censoring |
| **2** | **Survival modeling (this notebook)** | **Weibull parameters per component, validated hazard ratios for drivers** |
| 3 | Fleet cost forecast | Quarterly failure and cost forecast, backtested |
| 4 | New generation forecasting | Hierarchical Bayesian model for sparse data |

### Implementation note
Kaplan Meier and Weibull are written **from scratch** with NumPy and SciPy so every step is visible and testable. Cox PH uses the `lifelines` library (`pip install lifelines`), the standard Python survival package.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gamma as gamma_fn
from scipy.stats import norm

DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")

COMPONENTS = ["comp1", "comp2", "comp3", "comp4"]
COLORS = {"comp1": "#534AB7", "comp2": "#D85A30", "comp3": "#1D9E75", "comp4": "#BA7517"}
RNG = np.random.default_rng(42)

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

## 2. Load the lifetime table and handle left truncation
Some lifetimes **started in 2014**, before the observation window. We only have them in the table because they survived into 2015. A part that failed in 2014 was never recorded as a failure, so it is silently missing.

That is **left truncation**, and ignoring it makes parts look more reliable than they are: the early failures were filtered out before we looked.

The fix is to record, for every lifetime, the age at which we **started watching** it:

$$\text{entry} = \max(0,\ \text{OBS\_START} - \text{start})$$

A part installed on 2014-12-01 with the window opening 2015-01-01 enters at age 31 days. It only counts as "at risk" from day 31 onward. Every model below uses this column.

In [ ]:
lifetimes = pd.read_csv(DATA_PROCESSED / "component_lifetimes.csv", parse_dates=["start", "end"])
telemetry = pd.read_csv(DATA_RAW / "PdM_telemetry.csv", parse_dates=["datetime"])
errors = pd.read_csv(DATA_RAW / "PdM_errors.csv", parse_dates=["datetime"])

OBS_START = telemetry["datetime"].min()
OBS_END = telemetry["datetime"].max()

lifetimes["entry"] = ((OBS_START - lifetimes["start"]).dt.total_seconds() / 86_400).clip(lower=0)

# A lifetime must be observed for some positive time after entry
lifetimes = lifetimes[lifetimes["duration_days"] > lifetimes["entry"]].reset_index(drop=True)

print(f"{len(lifetimes):,} lifetimes  |  {lifetimes['event'].sum():,} failures")
print(f"Left truncated (started before window): {(lifetimes['entry'] > 0).mean():.1%}")
lifetimes[["machineID", "comp", "duration_days", "entry", "event", "ended_by", "model", "age"]].head()

## 3. Kaplan Meier: survival without assuming a distribution

### How it works (numerical example)
Five parts, sorted by lifetime:

| Part | Days | Ended by |
|---|---|---|
| A | 30 | failure |
| B | 50 | censored |
| C | 70 | failure |
| D | 90 | failure |
| E | 120 | censored |

At each **failure** time, multiply survival by (1 minus failures / parts still at risk):

| Time | At risk | Failures | S(t) |
|---|---|---|---|
| 30 | 5 | 1 | 1 × (1 − 1/5) = **0.800** |
| 70 | 3 | 1 | 0.800 × (1 − 1/3) = **0.533** |
| 90 | 2 | 1 | 0.533 × (1 − 1/2) = **0.267** |

Part B leaves the risk set at day 50 without dragging survival down, but it **did** count in the denominator at day 30. That is how censored parts contribute information. Dropping B and E instead would give S(90) = 0: every part fails by day 90, which the data does not say.

With left truncation, a part is in the risk set only between its `entry` age and its `duration`.

### Confidence band
Greenwood's formula gives the variance of S(t). We build the 95% CI (Confidence Interval) on the log(−log S) scale so the band stays between 0 and 1.

In [ ]:
def kaplan_meier(durations, events, entry=None) -> pd.DataFrame:
    d = np.asarray(durations, float)
    e = np.asarray(events, int)
    en = np.zeros_like(d) if entry is None else np.asarray(entry, float)

    times = np.unique(d[e == 1])
    at_risk = np.array([((en < t) & (d >= t)).sum() for t in times])
    deaths = np.array([((d == t) & (e == 1)).sum() for t in times])

    surv = np.cumprod(1 - deaths / at_risk)
    with np.errstate(divide="ignore", invalid="ignore"):
        greenwood = np.cumsum(deaths / (at_risk * (at_risk - deaths)))
        se_loglog = np.sqrt(greenwood) / np.abs(np.log(surv))
        z = norm.ppf(0.975)
        lo = surv ** np.exp(z * se_loglog)
        hi = surv ** np.exp(-z * se_loglog)

    out = pd.DataFrame({"time": times, "at_risk": at_risk, "failures": deaths,
                        "survival": surv, "ci_low": lo, "ci_high": hi})
    start = pd.DataFrame({"time": [0.0], "at_risk": [np.nan], "failures": [0],
                          "survival": [1.0], "ci_low": [1.0], "ci_high": [1.0]})
    return pd.concat([start, out], ignore_index=True).fillna({"ci_low": 0.0, "ci_high": 1.0})


def km_median(km: pd.DataFrame) -> float:
    below = km.loc[km["survival"] <= 0.5, "time"]
    return float(below.iloc[0]) if len(below) else np.nan


# Verify against the hand worked example above
toy = kaplan_meier([30, 50, 70, 90, 120], [1, 0, 1, 1, 0])
assert np.allclose(toy["survival"].values, [1.0, 0.8, 0.8 * 2 / 3, 0.8 * 2 / 3 * 0.5])
toy.round(3)

The toy result matches the hand calculation, so the function is correct. Now apply it to each component.

In [ ]:
km_results = {}
fig, ax = plt.subplots(figsize=(10, 5))
for comp in COMPONENTS:
    sub = lifetimes[lifetimes["comp"] == comp]
    km = kaplan_meier(sub["duration_days"], sub["event"], sub["entry"])
    km_results[comp] = km
    ax.step(km["time"], km["survival"], where="post", color=COLORS[comp], label=comp)
    ax.fill_between(km["time"], km["ci_low"], km["ci_high"], step="post", color=COLORS[comp], alpha=0.12)

ax.set(title="Kaplan Meier survival by component (95% CI)", xlabel="Component age (days)",
       ylabel="Probability still working", ylim=(0, 1.02))
ax.legend()
plt.show()

pd.DataFrame({
    comp: {"lifetimes": int((lifetimes["comp"] == comp).sum()),
           "failures": int(lifetimes.loc[lifetimes["comp"] == comp, "event"].sum()),
           "median_life_days": km_median(km),
           "S(90 days)": float(np.interp(90, km["time"], km["survival"])),
           "S(180 days)": float(np.interp(180, km["time"], km["survival"]))}
    for comp, km in km_results.items()
}).T.round(3)

**Reading the curves.** A curve that drops fast early means infant mortality; a curve that stays flat and then falls off a cliff means wear out. Where the curve never crosses 0.5, the median life is beyond what the data can see (shown as NaN), which is common when most lifetimes are censored.

### Does the platform matter?
Split each component's curve by machine model. If the curves separate clearly, a single fleet wide failure rate will misprice contracts for some platforms.

In [ ]:
models = sorted(lifetimes["model"].unique())
fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(15, 3.8), sharey=True)
for ax, comp in zip(axes, COMPONENTS):
    for model in models:
        sub = lifetimes[(lifetimes["comp"] == comp) & (lifetimes["model"] == model)]
        if sub["event"].sum() < 3:
            continue
        km = kaplan_meier(sub["duration_days"], sub["event"], sub["entry"])
        ax.step(km["time"], km["survival"], where="post", label=model)
    ax.set(title=comp, xlabel="Age (days)", ylim=(0, 1.02))
axes[0].set_ylabel("Probability still working")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

Kaplan Meier is honest but limited: it cannot extrapolate beyond the oldest observed part, and it does not summarize failure behavior in a few numbers a service planner can use. Weibull does both.

**Look at where the curves start.** Survival sits at 1.0 for the first few weeks: almost nothing fails young. That is a **failure free period**, and it shapes which Weibull model we fit next.

## 4. Weibull fits: the shape of failure

### The model
The 3 parameter Weibull adds a location γ (gamma), the minimum life before any failure can occur:

$$S(t) = \exp\!\left[-\left(\frac{t - \gamma}{\eta}\right)^{\beta}\right] \text{ for } t > \gamma, \qquad S(t) = 1 \text{ for } t \le \gamma$$

$$h(t) = \frac{\beta}{\eta}\left(\frac{t - \gamma}{\eta}\right)^{\beta - 1} \text{ for } t > \gamma$$

Setting γ = 0 gives the standard 2 parameter Weibull.

| β (shape) | Hazard over age | Physical meaning | Service action |
|---|---|---|---|
| β < 1 | falling | infant mortality: manufacturing or install defects | burn in, supplier quality, install checklists |
| β ≈ 1 | flat | random failures (reduces to constant AFR) | stock spares, preventive swaps do not help |
| β > 1 | rising | wear out | preventive replacement before the steep part |

η (scale) is how far past γ it takes for 63.2% of parts to fail.

### Why γ matters (numerical example)
Take a part with γ = 30 days, β = 1.5, η = 120 days. Its B10 (age by which 10% have failed) is 30 + 120 × 0.1054^(1/1.5) = **57 days**, and it has a 0% chance of failing in its first 30 days. A 2 parameter fit to the same data cannot put zero probability anywhere, so it spreads failures into the first month, pulls B10 earlier, and overstates failures for freshly replaced parts. On a fleet where preventive swaps keep resetting parts to age 0, that bias repeats every quarter.

### Fitting with censoring and truncation: MLE (Maximum Likelihood Estimation)
Each lifetime adds to the log likelihood:
1. **Failure at t:** log h(t) + log S(t). The part survived to t, then failed.
2. **Censored at t:** log S(t) only. It survived at least to t.
3. **Truncation:** subtract log S(entry). We condition on survival to the age we started watching.

We optimize over log β, log η, and a squashed γ that is forced to stay between 0 and the earliest observed failure (a part cannot fail before its minimum life).

In [ ]:
def weibull_S(t, beta, eta, gamma=0.0):
    z = np.clip(np.asarray(t, float) - gamma, 0, None) / eta
    return np.exp(-z ** beta)


def weibull_h(t, beta, eta, gamma=0.0):
    z = np.clip(np.asarray(t, float) - gamma, 0, None) / eta
    return np.where(z > 0, (beta / eta) * z ** (beta - 1), 0.0)


def _unpack(x, gamma_max):
    beta, eta = np.exp(x[0]), np.exp(x[1])
    gamma = gamma_max / (1 + np.exp(-x[2])) if len(x) == 3 else 0.0
    return beta, eta, gamma


def weibull_negloglik(x, t, event, entry, gamma_max):
    beta, eta, gamma = _unpack(x, gamma_max)
    z = np.clip(t - gamma, 1e-12, None) / eta
    z_entry = np.clip(entry - gamma, 0, None) / eta
    log_h = np.log(beta / eta) + (beta - 1) * np.log(z)
    return -(np.sum(event * log_h) - np.sum(z ** beta) + np.sum(z_entry ** beta))


def fit_weibull(t, event, entry=None, location=False) -> dict:
    t = np.asarray(t, float)
    event = np.asarray(event, float)
    entry = np.zeros_like(t) if entry is None else np.asarray(entry, float)
    gamma_max = 0.99 * t[event == 1].min()
    x0 = [0.0, np.log(t.mean())] + ([0.0] if location else [])
    res = minimize(weibull_negloglik, x0, args=(t, event, entry, gamma_max), method="Nelder-Mead",
                   options={"maxiter": 5_000, "xatol": 1e-6, "fatol": 1e-8})
    if not res.success:
        raise RuntimeError(res.message)
    beta, eta, gamma = _unpack(res.x, gamma_max)
    k = len(x0)
    return {"beta": float(beta), "eta": float(eta), "gamma": float(gamma), "gamma_max": float(gamma_max),
            "loglik": float(-res.fun), "AIC": float(2 * k + 2 * res.fun)}


# Verify 1: standard Weibull, beta = 1.4, eta = 300, heavy random censoring
life = 300 * RNG.weibull(1.4, 5_000)
censor_at = RNG.uniform(0, 600, 5_000)
t_obs, ev_obs = np.minimum(life, censor_at), (life <= censor_at).astype(int)
fit2 = fit_weibull(t_obs, ev_obs)
print(f"2p check  true beta 1.4, eta 300          |  fitted beta {fit2['beta']:.3f}, eta {fit2['eta']:.1f}")
assert abs(fit2["beta"] - 1.4) < 0.1 and abs(fit2["eta"] - 300) < 20

# Verify 2: failure free period, gamma = 30, beta = 1.5, eta = 120
life = 30 + 120 * RNG.weibull(1.5, 5_000)
censor_at = RNG.uniform(0, 300, 5_000)
t_obs, ev_obs = np.minimum(life, censor_at), (life <= censor_at).astype(int)
fit3 = fit_weibull(t_obs, ev_obs, location=True)
print(f"3p check  true beta 1.5, eta 120, gamma 30 |  fitted beta {fit3['beta']:.3f}, "
      f"eta {fit3['eta']:.1f}, gamma {fit3['gamma']:.1f}")
assert abs(fit3["beta"] - 1.5) < 0.15 and abs(fit3["eta"] - 120) < 10 and abs(fit3["gamma"] - 30) < 3

The fitter recovers the true parameters in both cases, even with heavy censoring.

### 2 parameter or 3 parameter? Let the data decide, with a safeguard
AIC (Akaike Information Criterion) = 2k − 2 log L rewards fit and penalizes the extra parameter. We use the 3 parameter model only if it lowers AIC by more than 2, the usual threshold for a meaningful improvement.

**The safeguard.** The 3 parameter Weibull has a known trap. When β < 1, the likelihood grows without limit as γ approaches the earliest observed failure, so the optimizer happily pushes γ against that bound and reports a "better" fit that describes nothing real. The symptom is unmistakable: γ lands at the bound. For example, if the earliest failure is at 26 days, the bound here is 0.99 × 26 = 25.74, and a fit returning γ = 25.74 with β = 0.92 is an artifact, not evidence of infant mortality after a failure free period.

So a 3 parameter fit is accepted only if all three hold:
1. AIC improves by more than 2.
2. β ≥ 1 (the region where the 3 parameter MLE is well behaved).
3. γ sits clearly inside its range (below 95% of the bound).

Otherwise the 2 parameter model wins. The table shows every check so the decision can be audited.

In [ ]:
AIC_THRESHOLD = 2

choice_rows = []
for comp in COMPONENTS:
    sub = lifetimes[lifetimes["comp"] == comp]
    f2 = fit_weibull(sub["duration_days"], sub["event"], sub["entry"], location=False)
    f3 = fit_weibull(sub["duration_days"], sub["event"], sub["entry"], location=True)
    delta = f2["AIC"] - f3["AIC"]
    valid_3p = (f3["beta"] >= 1) and (f3["gamma"] < 0.95 * f3["gamma_max"])
    choice_rows.append({"comp": comp, "AIC_2p": f2["AIC"], "AIC_3p": f3["AIC"], "AIC_improvement": delta,
                        "beta_2p": f2["beta"], "beta_3p": f3["beta"], "gamma_3p": f3["gamma"],
                        "gamma_bound": f3["gamma_max"], "valid_3p": valid_3p,
                        "chosen": "3p" if (delta > AIC_THRESHOLD and valid_3p) else "2p"})

model_choice = pd.DataFrame(choice_rows).set_index("comp")
model_choice.round(2)

Notice how β changes between the two fits. When a failure free period exists, the 2 parameter model has to stretch its shape to fake one, so its β is not directly comparable to the 3 parameter β. Always report β together with the model it came from.

### Fit the chosen model, with bootstrap CIs
Lifetimes on the same machine share its conditions, so they are not independent. We use a **cluster bootstrap**: resample whole machines with replacement, refit, repeat. The spread of the refitted parameters is the uncertainty.

Each bootstrap resample repeats the **whole procedure**, including the 2 versus 3 parameter choice and its safeguard. When the choice is close, some resamples pick one model and some the other, and that model uncertainty flows into the intervals honestly. Forcing the 3 parameter model on every resample would instead let degenerate fits (β far below 1, γ at the bound) leak into the CI. The column `share_3p` reports how often the resamples chose the 3 parameter model: near 0 or 1 means the choice is stable; in between means it is a close call.

The shape verdict is only declared when the entire 95% CI (Confidence Interval) of β sits on one side of 1. Otherwise the data cannot distinguish wear out from random failure, and we say so.

In [ ]:
N_BOOT = 200


def fit_selected(t, event, entry) -> dict:
    f2 = fit_weibull(t, event, entry, location=False)
    try:
        f3 = fit_weibull(t, event, entry, location=True)
    except RuntimeError:
        return {**f2, "model": "2p"}
    valid_3p = f3["beta"] >= 1 and f3["gamma"] < 0.95 * f3["gamma_max"]
    if f2["AIC"] - f3["AIC"] > AIC_THRESHOLD and valid_3p:
        return {**f3, "model": "3p"}
    return {**f2, "model": "2p"}


def bootstrap_weibull(sub: pd.DataFrame, n_boot: int = N_BOOT) -> pd.DataFrame:
    groups = {m: g for m, g in sub.groupby("machineID")}
    ids = np.array(list(groups))
    draws = []
    for _ in range(n_boot):
        sample = pd.concat([groups[m] for m in RNG.choice(ids, len(ids), replace=True)])
        try:
            draws.append(fit_selected(sample["duration_days"], sample["event"], sample["entry"]))
        except (RuntimeError, ValueError):
            continue
    return pd.DataFrame(draws)


def shape_verdict(lo: float, hi: float) -> str:
    if lo > 1:
        return "wear out"
    if hi < 1:
        return "infant mortality"
    return "not distinguishable from random"


rows = []
for comp in COMPONENTS:
    sub = lifetimes[lifetimes["comp"] == comp]
    fit = fit_selected(sub["duration_days"], sub["event"], sub["entry"])
    assert fit["model"] == model_choice.loc[comp, "chosen"]
    boot = bootstrap_weibull(sub)
    b, e, g = fit["beta"], fit["eta"], fit["gamma"]
    rows.append({"comp": comp, "model": fit["model"], "share_3p": (boot["model"] == "3p").mean(),
                 "beta": b, "beta_lo": boot["beta"].quantile(0.025), "beta_hi": boot["beta"].quantile(0.975),
                 "eta": e, "eta_lo": boot["eta"].quantile(0.025), "eta_hi": boot["eta"].quantile(0.975),
                 "gamma": g, "gamma_lo": boot["gamma"].quantile(0.025), "gamma_hi": boot["gamma"].quantile(0.975),
                 "mean_life_days": g + e * gamma_fn(1 + 1 / b),
                 "B10_days": g + e * (-np.log(0.9)) ** (1 / b),
                 "shape": shape_verdict(boot["beta"].quantile(0.025), boot["beta"].quantile(0.975))})

weibull = pd.DataFrame(rows).set_index("comp")
weibull.round(2)

**Columns:**
1. `beta`, `eta`, `gamma` with their 95% bootstrap bounds (`gamma` is 0 for 2 parameter fits).
2. `mean_life_days` = γ + η · Γ(1 + 1/β), the Weibull equivalent of MTBF (Mean Time Between Failures).
3. `B10_days` = γ + η · (−ln 0.9)^(1/β): the age by which 10% of parts have failed. Reliability engineers use it to set preventive swap intervals.

### Does the chosen model fit? Overlay on Kaplan Meier
The coloured line is the chosen model and the dashed line is the 2 parameter fit for comparison. If the chosen curve stays inside the Kaplan Meier band, the summary is trustworthy.

In [ ]:
fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(15, 3.8), sharey=True)
for ax, comp in zip(axes, COMPONENTS):
    km = km_results[comp]
    b, e, g = weibull.loc[comp, ["beta", "eta", "gamma"]]
    sub = lifetimes[lifetimes["comp"] == comp]
    f2 = fit_weibull(sub["duration_days"], sub["event"], sub["entry"], location=False)
    grid = np.linspace(0, km["time"].max(), 300)
    ax.step(km["time"], km["survival"], where="post", color="grey", label="Kaplan Meier")
    ax.fill_between(km["time"], km["ci_low"], km["ci_high"], step="post", color="grey", alpha=0.15)
    ax.plot(grid, weibull_S(grid, f2["beta"], f2["eta"]), color="black", ls="--", lw=1, label="2p Weibull")
    ax.plot(grid, weibull_S(grid, b, e, g), color=COLORS[comp], lw=2,
            label=f"chosen ({weibull.loc[comp, 'model']}) β={b:.2f}")
    ax.set(title=comp, xlabel="Age (days)", ylim=(0, 1.02))
    ax.legend(fontsize=7)
axes[0].set_ylabel("Probability still working")
plt.tight_layout()
plt.show()

### Hazard curves
The hazard is the most decision relevant view: it shows how risky a part of a given age is **right now**. A flat line is the constant AFR from Part 1; a rising line means old parts deserve more spares and earlier preventive swaps. With a 3 parameter fit the hazard is exactly zero until γ.

In [ ]:
grid = np.linspace(1, lifetimes["duration_days"].quantile(0.95), 300)
fig, ax = plt.subplots()
for comp in COMPONENTS:
    b, e, g = weibull.loc[comp, ["beta", "eta", "gamma"]]
    ax.plot(grid, weibull_h(grid, b, e, g) * 365, color=COLORS[comp], label=f"{comp} (β={b:.2f}, γ={g:.0f}d)")
ax.set(title="Weibull hazard by component age", xlabel="Component age (days)",
       ylabel="Failure rate (per year, annualized)")
ax.legend()
plt.show()

## 5. Business translation: expected failures next quarter
The units installed **today** are the lifetimes that ended by `end_of_data`, plus a fresh part (age 0) for any position whose part was replaced at the very last timestamp. Their current age is `duration_days`. For each one, the probability of failing in the next 90 days given that it has survived this long is:

$$P(\text{fail in next } h \mid \text{age } a) = 1 - \frac{S(a + h)}{S(a)}$$

Summing those probabilities gives the expected number of failures. Because each unit either fails or not, the total follows a Poisson binomial distribution; we simulate it to get an 80% range rather than a single number.

The reference point beside it is the **constant AFR forecast** from Part 1 (AFR × 90 / 365 × units installed), which ignores age. Because every machine carries one unit of each component all year, total exposure is roughly units × 365 days, so this number is also the observed 2015 run rate scaled to 90 days. A Weibull forecast far from it needs an explanation.

**What the Weibull number leaves out.** It counts at most one failure per installed part and assumes no preventive swaps. Preventive swaps push the true count down; parts that fail and are replaced can fail again within the quarter, which pushes it up. Part 3 simulates both.

In [ ]:
HORIZON = 90
N_SIM = 10_000

# Parts in service when the data ends. A part replaced exactly at the last timestamp has a zero length
# successor that Part 1 could not record, so that position gets a fresh part at age 0.
last = lifetimes.sort_values("end").groupby(["machineID", "comp"]).tail(1)
fresh = last[(last["end"] == OBS_END) & (last["ended_by"] != "end_of_data")].assign(duration_days=0.0)
installed = pd.concat([lifetimes[lifetimes["ended_by"] == "end_of_data"], fresh], ignore_index=True)

rows = []
for comp in COMPONENTS:
    units = installed[installed["comp"] == comp]
    b, e, g = weibull.loc[comp, ["beta", "eta", "gamma"]]
    age = units["duration_days"].to_numpy()
    p = 1 - weibull_S(age + HORIZON, b, e, g) / weibull_S(age, b, e, g)

    sims = (RNG.random((N_SIM, len(p))) < p).sum(axis=1)

    comp_life = lifetimes[lifetimes["comp"] == comp]
    afr = comp_life["event"].sum() / (comp_life["duration_days"] - comp_life["entry"]).sum() * 365

    rows.append({"comp": comp, "units_installed": len(units), "mean_age_days": age.mean(),
                 "expected_failures_weibull": p.sum(),
                 "p10": np.percentile(sims, 10), "p90": np.percentile(sims, 90),
                 "expected_failures_constant_AFR": afr * HORIZON / 365 * len(units)})

next_q = pd.DataFrame(rows).set_index("comp")
next_q.loc["total"] = next_q.sum(numeric_only=True)
next_q.loc["total", ["mean_age_days", "p10", "p90"]] = np.nan
next_q.round(1)

**How to read the gap.** Where the Weibull and constant AFR forecasts differ, the age mix of the fleet is doing the work. If the Weibull number is higher, the installed parts are older than average on a wear out curve, and a constant rate model would understock spares and under reserve for the quarter.

The p10 to p90 range is the start of the uncertainty story that Part 3 turns into dollars.

## 6. Cox PH: what drives failure risk?
Weibull describes **how** each component fails. Cox PH explains **why some machines fail more** than others.

### The model
$$h(t \mid x) = h_0(t) \cdot \exp(\beta_1 x_1 + \beta_2 x_2 + \dots)$$

$h_0(t)$ is a baseline hazard left completely unspecified; covariates scale it up or down by a constant multiplier, the HR (Hazard Ratio) = exp(coefficient).

### Numerical example
If machine age has HR = 1.04 per year, a machine 10 years older has a hazard 1.04¹⁰ = **1.48 times** higher at every component age. If a fleet of average age expects 30 failures a quarter, shifting it 10 years older pushes that toward roughly 44, before any other change.

### Three design rules for a trustworthy driver model

**Rule 1: covariates must be measured before the outcomes they explain.**
Averaging telemetry over the whole year leaks the future into the past: a lifetime that ended in March would be scored with sensor data from April to December. It also reverses cause and effect. A machine that spends the two weeks before each of its 8 failures in a degraded state with voltage 20 units high has 8 × 14 = 112 abnormal days out of 365. Its yearly mean voltage rises by 20 × 112 / 365 ≈ **6.1** purely *because* it failed, and the model would then "discover" that voltage causes failure.

The fix is a clean time split. Machine usage profiles come from the **first half** of 2015 only, and the Cox model is fit on lifetimes that **start in the second half**. Every covariate is then measured strictly before the lifetime it explains, exactly as it would be at forecast time. A bonus: those lifetimes all start inside the window, so there is no left truncation and the proportional hazards test runs directly.

**Rule 2: one model per component.**
A platform can be fine for one module and terrible for another. A single pooled model estimates one HR per platform across all components and blends those different stories into a number that describes none of them. Fitting each component separately lets every component have its own drivers.

If a platform has **zero failures** of a component, no HR can be estimated for it (the best fit is a hazard of exactly zero). Those lifetimes are reported as a finding and excluded from that component's model.

**Rule 3: validate with grouped cross validation, not one random split.**
With 20 held out machines and 30 to 40 failures, one split can land anywhere between a coin flip and excellent by luck alone. Five folds, each holding out a different 20% of machines, give a mean and a spread.

### Covariates
| Covariate | Scale | Question it answers |
|---|---|---|
| `age` | per year of machine age | do older tools break more? |
| `model_*` | vs the first platform with failures (baseline) | do some platforms break more? |
| `volt_z`, `rotate_z`, `pressure_z`, `vibration_z` | per 1 SD (Standard Deviation) of the machine's first half mean reading | does heavier usage drive failure? |
| `errors_z` | per 1 SD of the machine's first half error rate | do alarm prone tools fail more? |

In [ ]:
SPLIT = OBS_START + pd.Timedelta(days=181)
h1_days = (SPLIT - OBS_START).days

h1_tel = telemetry[telemetry["datetime"] < SPLIT]
machine_usage = h1_tel.groupby("machineID")[["volt", "rotate", "pressure", "vibration"]].mean()
h1_errors = errors[(errors["datetime"] >= OBS_START) & (errors["datetime"] < SPLIT)]
machine_usage["errors"] = (h1_errors.groupby("machineID").size()
                           .reindex(machine_usage.index, fill_value=0) / h1_days * 30)

usage_z = (machine_usage - machine_usage.mean()) / machine_usage.std()
usage_z.columns = [f"{c}_z" for c in usage_z.columns]
USAGE_COLS = list(usage_z.columns)

cox_base = (lifetimes[lifetimes["start"] >= SPLIT]
            .merge(usage_z, left_on="machineID", right_index=True, how="left"))
assert cox_base[USAGE_COLS].notna().all().all(), "Missing usage covariates"
assert (cox_base["entry"] == 0).all(), "Second half lifetimes should not be truncated"

print(f"Usage profile window: {OBS_START.date()} to {SPLIT.date()}  |  Cox lifetimes start on or after {SPLIT.date()}")
cox_base.groupby("comp").agg(lifetimes=("event", "size"), failures=("event", "sum"))

### Failures by platform in the modeling window
Any zero in this table is a platform that never failed that component. It gets reported, not modeled.

In [ ]:
events_by_model = pd.crosstab(cox_base["comp"], cox_base["model"], values=cox_base["event"], aggfunc="sum").fillna(0)
events_by_model.astype(int)

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from lifelines.utils import concordance_index

PENALIZER = 0.05


def prepare_component(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:
    failures_by_model = df.groupby("model")["event"].sum()
    keep = failures_by_model[failures_by_model > 0].index
    excluded = [{"comp": df["comp"].iloc[0], "model": m, "lifetimes": int((df["model"] == m).sum()), "failures": 0}
                for m in failures_by_model.index if m not in keep]
    d = df[df["model"].isin(keep)]
    dummies = pd.get_dummies(d["model"], prefix="model", drop_first=True, dtype=float)
    d = pd.concat([d[["duration_days", "event", "machineID", "age"] + USAGE_COLS], dummies], axis=1)
    return d.reset_index(drop=True), excluded


def fit_cox(d: pd.DataFrame) -> CoxPHFitter:
    return CoxPHFitter(penalizer=PENALIZER).fit(d.drop(columns="machineID"),
                                                duration_col="duration_days", event_col="event")


cox_data, fits, hr_rows, excluded_rows = {}, {}, [], []
for comp in COMPONENTS:
    d, excluded = prepare_component(cox_base[cox_base["comp"] == comp])
    cox_data[comp], fits[comp] = d, fit_cox(d)
    excluded_rows += excluded
    s = fits[comp].summary
    for cov, r in s.iterrows():
        hr_rows.append({"comp": comp, "covariate": cov, "HR": r["exp(coef)"],
                        "HR_lo": r["exp(coef) lower 95%"], "HR_hi": r["exp(coef) upper 95%"], "p_value": r["p"]})

hr = pd.DataFrame(hr_rows)
hr["significant"] = (hr["HR_lo"] > 1) | (hr["HR_hi"] < 1)

if excluded_rows:
    print("Platforms with zero failures of a component (reported, not modeled):")
    display(pd.DataFrame(excluded_rows))

# Wide view: HR per component, * marks a 95% CI that excludes 1
wide = hr.assign(cell=hr["HR"].map("{:.2f}".format) + np.where(hr["significant"], "*", ""))
wide.pivot(index="covariate", columns="comp", values="cell").fillna("")

**Reading the table.** HR = 1.30* on `vibration_z` for comp1 would mean a machine one SD above average first half vibration has 30% higher comp1 failure hazard at every component age, and the star says the 95% CI excludes 1. An empty cell means that covariate does not exist for that component (for example a platform with zero failures).

A forest plot per component makes the drivers readable at a glance: points right of the dashed line raise risk, points left lower it.

In [ ]:
fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(16, 4.5))
for ax, comp in zip(axes, COMPONENTS):
    sub = hr[hr["comp"] == comp].sort_values("HR")
    y = np.arange(len(sub))
    ax.errorbar(sub["HR"], y, xerr=[sub["HR"] - sub["HR_lo"], sub["HR_hi"] - sub["HR"]],
                fmt="o", color=COLORS[comp], capsize=3)
    ax.axvline(1, color="grey", ls="--")
    ax.set_yticks(y, sub["covariate"], fontsize=8)
    ax.set(xscale="log", title=comp, xlabel="Hazard ratio (log scale)")
plt.suptitle("Failure risk drivers by component (95% CI)")
plt.tight_layout()
plt.show()

### Validation 1: ranking power, 5 fold grouped cross validation
The concordance index (C index) asks: for two parts where one failed first, did the model assign it the higher risk? 0.5 is a coin flip, 0.7 is useful, above 0.8 is strong.

Each fold holds out a different 20% of machines, so no machine appears in both training and scoring. We report the mean and SD across folds. A model is marked **usable** only if the mean C index minus one SD still clears 0.5; otherwise its apparent skill is within the noise.

Note that covariates are per machine, so every lifetime on the same machine gets the same risk score. The C index counts those as ties, which caps how high it can go. That is a property of machine level drivers, not a bug.

In [ ]:
N_FOLDS = 5


def grouped_cv(d: pd.DataFrame, n_folds: int = N_FOLDS) -> np.ndarray:
    folds = np.array_split(RNG.permutation(d["machineID"].unique()), n_folds)
    scores = []
    for test_ids in folds:
        train_d, test_d = d[~d["machineID"].isin(test_ids)], d[d["machineID"].isin(test_ids)]
        if test_d["event"].sum() < 2:
            continue
        risk = fit_cox(train_d).predict_partial_hazard(test_d.drop(columns="machineID")).to_numpy().ravel()
        scores.append(concordance_index(test_d["duration_days"], -risk, test_d["event"]))
    return np.array(scores)


cv_rows = []
for comp in COMPONENTS:
    scores = grouped_cv(cox_data[comp])
    enough = len(scores) >= 3
    cv_rows.append({"comp": comp, "in_sample_C": fits[comp].concordance_index_,
                    "cv_mean_C": scores.mean() if enough else np.nan,
                    "cv_sd_C": scores.std(ddof=1) if enough else np.nan, "folds": len(scores)})

cox_validation = pd.DataFrame(cv_rows).set_index("comp")
cox_validation["verdict"] = np.select(
    [cox_validation["folds"] < 3, cox_validation["cv_mean_C"] - cox_validation["cv_sd_C"] > 0.5],
    ["insufficient data", "usable"], "not reliable")
cox_validation.round(3)

**How Part 3 uses this.** For components marked **usable**, the hazard ratios adjust each machine's failure probability. For components marked **not reliable** or **insufficient data** (fewer than 3 folds with at least 2 failures to score), Part 3 uses the component Weibull alone: adding multipliers that do not predict out of sample only adds noise to the forecast. A large gap between `in_sample_C` and `cv_mean_C` is overfitting.

### Validation 2: the proportional hazards assumption
Cox assumes each covariate multiplies the hazard by the **same** factor at every age. If, say, vibration matters only for old parts, that assumption fails. The test uses scaled Schoenfeld residuals; a small p value flags a covariate whose effect changes with component age.

Because the modeling lifetimes are not truncated, the test runs on the actual fitted models with no workaround.

In [ ]:
ph_rows = []
for comp in COMPONENTS:
    res = proportional_hazard_test(fits[comp], cox_data[comp].drop(columns="machineID"), time_transform="rank")
    for cov, p in zip(res.summary.index.get_level_values(0), res.summary["p"]):
        ph_rows.append({"comp": comp, "covariate": cov, "p": p})

ph = pd.DataFrame(ph_rows)
ph_wide = ph.pivot(index="covariate", columns="comp", values="p")
print("Covariates violating proportional hazards (p < 0.05):")
print(ph[ph["p"] < 0.05][["comp", "covariate", "p"]].round(4).to_string(index=False) if (ph["p"] < 0.05).any() else "  none")
ph_wide.round(3)

With 9 covariates and 4 components there are 36 tests, so about 2 will fall below 0.05 by chance even if the assumption holds everywhere. A flag deserves attention when p is very small or when the same covariate is flagged for several components. The standard remedies are to stratify on that covariate or to let its effect vary with age. Record flagged covariates in the README as known limitations.

## 7. Save outputs for Part 3
Part 3 needs the chosen Weibull model per component, the hazard ratios and their validation verdicts, and the machine usage profile that produced them.

In [ ]:
weibull.to_csv(DATA_PROCESSED / "weibull_params.csv")
hr.to_csv(DATA_PROCESSED / "cox_hazard_ratios.csv", index=False)
cox_validation.to_csv(DATA_PROCESSED / "cox_validation.csv")
machine_usage.to_csv(DATA_PROCESSED / "machine_usage_h1.csv")
next_q.to_csv(DATA_PROCESSED / "next_quarter_failure_exposure.csv")
print("Saved: weibull_params.csv, cox_hazard_ratios.csv, cox_validation.csv, "
      "machine_usage_h1.csv, next_quarter_failure_exposure.csv")

### Summary
1. Corrected for **left truncation**: parts installed before the window only count as at risk from the age we started watching them.
2. Built **Kaplan Meier** curves from scratch, verified against a hand worked example, with log(−log) Greenwood CIs.
3. Fit **2 and 3 parameter Weibull** models by MLE with censoring and truncation, verified on simulated data, and chose between them by AIC. The 3 parameter model captures the failure free period visible in the Kaplan Meier curves. Machine level cluster bootstrap CIs back every shape verdict.
4. Translated the fits into **expected failures next quarter** for the parts installed today, with a simulated p10 to p90 range, compared against the constant AFR baseline (equivalent to the observed run rate).
5. Fit **per component Cox PH** models with covariates measured strictly before the lifetimes they explain, reported platforms with zero failures instead of forcing estimates, validated with 5 fold grouped cross validation, and tested the proportional hazards assumption directly.

### Next: Part 3, fleet cost forecast
Simulate the replacement process of every installed part (failures, repeat failures within the quarter, and preventive swaps), apply hazard ratios only where they validated, attach a cost per event, and **backtest**: fit on the first part of the window, forecast the last quarter, and compare against what actually failed.